# Multi-Task Models: Head-to-Head Comparison on Ali-CCP

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/your-repo/rec_system_experimental/blob/main/notebooks/04_aliccp_cvr/04_comparison.ipynb)

---

## Learning Objectives

By the end of this notebook, you will:
1. Compare Naive CVR, ESMM, MMoE, and PLE on identical Ali-CCP data
2. Analyze sample selection bias mitigation across architectures
3. Evaluate business impact through conversion lift analysis
4. Compute statistical significance via bootstrap confidence intervals
5. Understand architecture-performance trade-offs for deployment decisions

## Prerequisites

- Completed Notebooks 01-03
- Results files at `data/aliccp/processed/`

## Table of Contents

1. [Load Results](#1-load-results)
2. [Comprehensive Metrics Comparison](#2-comprehensive-metrics-comparison)
3. [Sample Selection Bias Analysis](#3-sample-selection-bias-analysis)
4. [Business Impact Analysis](#4-business-impact-analysis)
5. [Statistical Significance](#5-statistical-significance)
6. [Architecture Feature Comparison](#6-architecture-feature-comparison)
7. [Latency-Accuracy Trade-off](#7-latency-accuracy-trade-off)
8. [Deployment Recommendations](#8-deployment-recommendations)
9. [Exercises](#exercises)
10. [Summary & Key Takeaways](#summary--key-takeaways)

In [ ]:
import os
import json
import time
import pickle
import warnings
from pathlib import Path
from collections import OrderedDict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.metrics import roc_auc_score, roc_curve

warnings.filterwarnings('ignore')

# Plotting
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 12
plt.style.use('seaborn-v0_8-whitegrid')

# Paths
DATA_DIR = Path('../../data/aliccp')
PROCESSED_DIR = DATA_DIR / 'processed'

# Color scheme for models
MODEL_COLORS = {
    'Naive CVR': '#bab0ac',
    'ESMM': '#4e79a7',
    'MMoE': '#f28e2b',
    'PLE': '#e15759'
}

## 1. Load Results

In [ ]:
# Load results from all notebooks
all_results = {}

# ESMM + Naive results
if (PROCESSED_DIR / 'esmm_results.json').exists():
    with open(PROCESSED_DIR / 'esmm_results.json') as f:
        esmm_data = json.load(f)
    all_results['Naive CVR'] = esmm_data.get('naive_cvr', {})
    all_results['ESMM'] = esmm_data.get('esmm', {})
    print('Loaded ESMM results')

# Advanced model results
if (PROCESSED_DIR / 'advanced_cvr_results.json').exists():
    with open(PROCESSED_DIR / 'advanced_cvr_results.json') as f:
        adv_data = json.load(f)
    all_results['MMoE'] = adv_data.get('mmoe', {})
    all_results['PLE'] = adv_data.get('ple', {})
    print('Loaded Advanced CVR results')

# Also try the combined results file
if (PROCESSED_DIR / 'aliccp_results.json').exists():
    with open(PROCESSED_DIR / 'aliccp_results.json') as f:
        combined = json.load(f)
    # Use combined results as fallback/override
    if 'naive_cvr' in combined:
        all_results.setdefault('Naive CVR', combined['naive_cvr'])
    if 'esmm' in combined:
        all_results.setdefault('ESMM', combined['esmm'])
    if 'mmoe' in combined:
        all_results.setdefault('MMoE', combined['mmoe'])
    if 'ple' in combined:
        all_results.setdefault('PLE', combined['ple'])
    print('Loaded combined results')

# Display results table
print('\n' + '=' * 70)
print('ALI-CCP MULTI-TASK CVR RESULTS')
print('=' * 70)
print(f'{"Model":<15} {"CTR AUC":>10} {"CVR AUC":>10} {"CTCVR AUC":>10} {"Params":>12}')
print('-' * 60)

for name in ['Naive CVR', 'ESMM', 'MMoE', 'PLE']:
    if name not in all_results:
        continue
    res = all_results[name]
    ctr = f'{res["ctr_auc"]:.4f}' if 'ctr_auc' in res else '    -'
    cvr = f'{res.get("cvr_auc", 0):.4f}'
    ctcvr = f'{res.get("ctcvr_auc", 0):.4f}'
    params = f'{res.get("params", 0):,}' if 'params' in res else '    -'
    print(f'{name:<15} {ctr:>10} {cvr:>10} {ctcvr:>10} {params:>12}')

## 2. Comprehensive Metrics Comparison

In [ ]:
# Fig 1: Multi-metric bar chart
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

model_names = [n for n in ['Naive CVR', 'ESMM', 'MMoE', 'PLE'] if n in all_results]
colors = [MODEL_COLORS[n] for n in model_names]

metrics_configs = [
    ('ctcvr_auc', 'CTCVR AUC\n(Primary Metric)', 0.50),
    ('cvr_auc', 'CVR AUC\n(Clicked Samples)', 0.50),
    ('ctr_auc', 'CTR AUC', 0.50),
]

for idx, (metric, title, ylim_bottom) in enumerate(metrics_configs):
    ax = axes[idx]
    vals = [all_results[n].get(metric, 0) for n in model_names]
    
    bars = ax.bar(model_names, vals, color=colors, edgecolor='white', linewidth=2)
    for bar, val in zip(bars, vals):
        if val > 0:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                    f'{val:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    ax.set_ylabel('AUC')
    ax.set_title(title, fontsize=13)
    if max(vals) > 0:
        ax.set_ylim(ylim_bottom, max(v for v in vals if v > 0) + 0.02)
    ax.grid(True, alpha=0.3, axis='y')
    ax.tick_params(axis='x', rotation=15)

plt.suptitle('Ali-CCP: Model Performance Comparison', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Fig 2: Radar chart
fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

radar_metrics = ['CTR AUC', 'CVR AUC', 'CTCVR AUC']
n_metrics = len(radar_metrics)
angles = np.linspace(0, 2 * np.pi, n_metrics, endpoint=False).tolist()
angles += angles[:1]  # close the polygon

for name in model_names:
    res = all_results[name]
    values = [
        res.get('ctr_auc', 0.5),
        res.get('cvr_auc', 0.5),
        res.get('ctcvr_auc', 0.5)
    ]
    # Normalize to [0, 1] range for radar
    min_val = 0.5
    max_val = 0.7
    norm_values = [(v - min_val) / (max_val - min_val) for v in values]
    norm_values += norm_values[:1]
    
    ax.plot(angles, norm_values, 'o-', linewidth=2, label=name,
            color=MODEL_COLORS[name], markersize=6)
    ax.fill(angles, norm_values, alpha=0.1, color=MODEL_COLORS[name])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(radar_metrics, fontsize=11)
ax.set_title('Multi-Metric Radar Comparison\n(Normalized 0.5-0.7 range)', fontsize=13, y=1.1)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=10)
plt.tight_layout()
plt.show()

## 3. Sample Selection Bias Analysis

> **Concept:** The key question is: do multi-task models (ESMM, MMoE, PLE) reduce the
> distribution shift between training and serving? We measure this by comparing conversion
> prediction quality on all impressions vs only clicked impressions.

In [ ]:
# Fig 3: SSB mitigation analysis
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# CVR AUC vs CTCVR AUC gap
ax = axes[0]
x = np.arange(len(model_names))
width = 0.35

cvr_vals = [all_results[n].get('cvr_auc', 0) for n in model_names]
ctcvr_vals = [all_results[n].get('ctcvr_auc', 0) for n in model_names]

bars1 = ax.bar(x - width/2, cvr_vals, width, label='CVR AUC (Clicked)',
               color=[MODEL_COLORS[n] for n in model_names], alpha=0.7, edgecolor='white')
bars2 = ax.bar(x + width/2, ctcvr_vals, width, label='CTCVR AUC (All)',
               color=[MODEL_COLORS[n] for n in model_names], edgecolor='white')

ax.set_xticks(x)
ax.set_xticklabels(model_names, rotation=15)
ax.set_ylabel('AUC')
ax.set_title('CVR vs CTCVR AUC\n(Closer = Less SSB)', fontsize=13)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

# SSB Gap (CVR - CTCVR)
ax = axes[1]
gaps = [cvr_vals[i] - ctcvr_vals[i] for i in range(len(model_names))]
bars = ax.bar(model_names, gaps, color=[MODEL_COLORS[n] for n in model_names],
              edgecolor='white', linewidth=2)
for bar, gap in zip(bars, gaps):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{gap:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_ylabel('CVR AUC - CTCVR AUC')
ax.set_title('SSB Gap\n(Smaller = Better Generalization)', fontsize=13)
ax.axhline(y=0, color='black', linewidth=0.5)
ax.grid(True, alpha=0.3, axis='y')
ax.tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

print('SSB Gap Analysis:')
for name, gap in zip(model_names, gaps):
    print(f'  {name}: CVR-CTCVR gap = {gap:+.4f}')

## 4. Business Impact Analysis

> **Concept:** Better conversion prediction translates to better ad ranking. We estimate
> the business impact by simulating how different models would rank ads and the resulting
> conversion lift.

> **Pro Tip:** In production, even a 0.01 AUC improvement can translate to millions in
> additional revenue when applied to billions of daily impressions.

In [ ]:
# Fig 4: CTCVR AUC improvement over baseline
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Improvement over Naive CVR
ax = axes[0]
naive_ctcvr = all_results.get('Naive CVR', {}).get('ctcvr_auc', 0.5)
improvements = []
for name in model_names:
    ctcvr = all_results[name].get('ctcvr_auc', 0)
    imp = (ctcvr - naive_ctcvr) / naive_ctcvr * 100 if naive_ctcvr > 0 else 0
    improvements.append(imp)

bars = ax.bar(model_names, improvements, color=[MODEL_COLORS[n] for n in model_names],
              edgecolor='white', linewidth=2)
for bar, imp in zip(bars, improvements):
    if imp != 0:
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + (0.3 if imp > 0 else -1.5),
                f'{imp:+.1f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_ylabel('Improvement (%)')
ax.set_title('CTCVR AUC Improvement over Naive CVR', fontsize=13)
ax.axhline(y=0, color='black', linewidth=0.5)
ax.grid(True, alpha=0.3, axis='y')
ax.tick_params(axis='x', rotation=15)

# Estimated conversion lift at top-K
ax = axes[1]
# Simulate: if we rank by CTCVR score, better AUC -> more conversions in top-K
# Simple model: AUC improvement -> proportional lift
top_k_impressions = [1000, 5000, 10000, 50000]

for name in ['ESMM', 'MMoE', 'PLE']:
    if name not in all_results:
        continue
    ctcvr_auc = all_results[name].get('ctcvr_auc', 0.5)
    # Estimated lift = (AUC - 0.5) / (naive_AUC - 0.5) * 100 - 100
    naive_lift = max(naive_ctcvr - 0.5, 0.001)
    model_lift = max(ctcvr_auc - 0.5, 0.001)
    relative_lift = [(model_lift / naive_lift - 1) * 100] * len(top_k_impressions)
    ax.plot(top_k_impressions, relative_lift, 'o-', label=name,
            color=MODEL_COLORS[name], linewidth=2, markersize=6)

ax.set_xlabel('Top-K Impressions Served')
ax.set_ylabel('Relative Conversion Lift (%)')
ax.set_title('Estimated Conversion Lift vs Naive CVR', fontsize=13)
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xscale('log')

plt.tight_layout()
plt.show()

## 5. Statistical Significance

> **Concept:** Bootstrap resampling gives us confidence intervals for AUC differences.
> We resample the validation set with replacement, compute AUC for each bootstrap sample,
> and check if the 95% CI for the difference excludes zero.

In [ ]:
# Bootstrap confidence intervals
# Since we don't have raw predictions here, we simulate based on AUC values
# In practice, you'd run this on actual predictions

np.random.seed(42)
n_bootstrap = 1000

print("=" * 70)
print("BOOTSTRAP CONFIDENCE INTERVALS (Simulated)")
print("=" * 70)

# Simulate bootstrap CIs based on known AUC values
# Standard error of AUC ~ sqrt(AUC*(1-AUC)/n_test)
n_test = 500_000  # approximate test size

bootstrap_results = {}
for name in model_names:
    ctcvr_auc = all_results[name].get('ctcvr_auc', 0.5)
    se = np.sqrt(ctcvr_auc * (1 - ctcvr_auc) / n_test)
    bootstrap_samples = np.random.normal(ctcvr_auc, se, n_bootstrap)
    ci_low = np.percentile(bootstrap_samples, 2.5)
    ci_high = np.percentile(bootstrap_samples, 97.5)
    bootstrap_results[name] = {
        'mean': ctcvr_auc, 'ci_low': ci_low, 'ci_high': ci_high,
        'se': se, 'samples': bootstrap_samples
    }
    print(f'{name:<15}: CTCVR AUC = {ctcvr_auc:.4f} '
          f'[{ci_low:.4f}, {ci_high:.4f}] (95% CI)')

# Pairwise significance tests
print(f'\nPairwise Differences (vs Naive CVR):')
naive_samples = bootstrap_results['Naive CVR']['samples']
for name in ['ESMM', 'MMoE', 'PLE']:
    if name not in bootstrap_results:
        continue
    diff_samples = bootstrap_results[name]['samples'] - naive_samples
    ci_low = np.percentile(diff_samples, 2.5)
    ci_high = np.percentile(diff_samples, 97.5)
    significant = ci_low > 0  # entire CI above 0
    sig_marker = '***' if significant else 'n.s.'
    print(f'  {name} - Naive: {np.mean(diff_samples):+.4f} '
          f'[{ci_low:+.4f}, {ci_high:+.4f}] {sig_marker}')

In [ ]:
# Fig 5: Bootstrap CI visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Forest plot of CTCVR AUC with CIs
ax = axes[0]
y_pos = range(len(model_names))
means = [bootstrap_results[n]['mean'] for n in model_names]
ci_lows = [bootstrap_results[n]['ci_low'] for n in model_names]
ci_highs = [bootstrap_results[n]['ci_high'] for n in model_names]
errors = [[m - l for m, l in zip(means, ci_lows)],
          [h - m for m, h in zip(means, ci_highs)]]

ax.errorbar(means, y_pos, xerr=errors, fmt='o', markersize=8,
            capsize=5, capthick=2, linewidth=2,
            color='#4e79a7')
ax.set_yticks(list(y_pos))
ax.set_yticklabels(model_names)
ax.set_xlabel('CTCVR AUC')
ax.set_title('CTCVR AUC with 95% Confidence Intervals', fontsize=13)
ax.grid(True, alpha=0.3, axis='x')

# Distribution of bootstrap differences (ESMM - Naive)
ax = axes[1]
for name, color in [('ESMM', '#4e79a7'), ('MMoE', '#f28e2b'), ('PLE', '#e15759')]:
    if name not in bootstrap_results:
        continue
    diff = bootstrap_results[name]['samples'] - bootstrap_results['Naive CVR']['samples']
    ax.hist(diff, bins=50, alpha=0.4, label=f'{name} - Naive', color=color)

ax.axvline(x=0, color='black', linestyle='--', linewidth=1, label='No difference')
ax.set_xlabel('CTCVR AUC Difference')
ax.set_ylabel('Count')
ax.set_title('Bootstrap Distribution of AUC Differences', fontsize=13)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Architecture Feature Comparison

In [ ]:
# Fig 6: Architecture feature comparison table
arch_features = pd.DataFrame({
    'Model': ['Naive CVR', 'ESMM', 'MMoE', 'PLE'],
    'Entire-Space Training': ['No', 'Yes', 'Yes', 'Yes'],
    'Shared Embeddings': ['N/A', 'Yes', 'Yes', 'Yes'],
    'Expert Networks': ['No', 'No', 'Shared', 'Shared + Task-Specific'],
    'Gating Mechanism': ['No', 'No', 'Per-Task', 'Per-Task'],
    'Progressive Layers': ['No', 'No', 'No', 'Yes'],
    'SSB Solution': ['None', 'CTR*CVR', 'CTR*CVR', 'CTR*CVR'],
    'CTCVR AUC': [all_results.get(n, {}).get('ctcvr_auc', 0)
                  for n in ['Naive CVR', 'ESMM', 'MMoE', 'PLE']],
    'Parameters': [all_results.get(n, {}).get('params', 'N/A')
                   for n in ['Naive CVR', 'ESMM', 'MMoE', 'PLE']],
})

print("=" * 90)
print("ARCHITECTURE FEATURE COMPARISON")
print("=" * 90)
print(arch_features.to_string(index=False))

In [ ]:
# Fig 7: Architecture features heatmap
fig, ax = plt.subplots(figsize=(10, 5))

feature_matrix = np.array([
    [0, 0, 0, 0, 0],  # Naive CVR
    [1, 1, 0, 0, 0],  # ESMM
    [1, 1, 1, 1, 0],  # MMoE
    [1, 1, 1, 1, 1],  # PLE
])

feature_names = ['Entire-Space\nTraining', 'Shared\nEmbeddings',
                 'Expert\nNetworks', 'Task\nGating', 'Progressive\nLayers']

sns.heatmap(feature_matrix, annot=True, fmt='d', cmap='YlOrRd',
            xticklabels=feature_names,
            yticklabels=['Naive CVR', 'ESMM', 'MMoE', 'PLE'],
            cbar_kws={'label': 'Has Feature (0=No, 1=Yes)'},
            linewidths=2, linecolor='white',
            ax=ax)
ax.set_title('Architecture Feature Matrix', fontsize=14)
plt.tight_layout()
plt.show()

## 7. Latency-Accuracy Trade-off

> **Pro Tip:** In production ad systems, model latency is as important as accuracy.
> Each additional millisecond in ad serving latency can reduce revenue by 1-2%.
> Complex models must justify their latency cost with meaningful AUC improvements.

In [ ]:
# Fig 8: Latency-accuracy trade-off
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Parameters vs CTCVR AUC
ax = axes[0]
for name in model_names:
    params = all_results[name].get('params', 0)
    ctcvr = all_results[name].get('ctcvr_auc', 0)
    if params > 0 and ctcvr > 0:
        ax.scatter(params / 1e6, ctcvr, s=200, c=MODEL_COLORS[name],
                   edgecolors='white', linewidth=2, zorder=5)
        ax.annotate(name, (params / 1e6, ctcvr),
                    textcoords='offset points', xytext=(10, 5),
                    fontsize=11, fontweight='bold')

ax.set_xlabel('Parameters (Millions)', fontsize=12)
ax.set_ylabel('CTCVR AUC', fontsize=12)
ax.set_title('Model Size vs Performance', fontsize=13)
ax.grid(True, alpha=0.3)

# Relative complexity vs improvement
ax = axes[1]
complexity_levels = {'Naive CVR': 1, 'ESMM': 2, 'MMoE': 3, 'PLE': 4}
for name in model_names:
    ctcvr = all_results[name].get('ctcvr_auc', 0)
    comp = complexity_levels.get(name, 0)
    if ctcvr > 0:
        ax.scatter(comp, ctcvr, s=200, c=MODEL_COLORS[name],
                   edgecolors='white', linewidth=2, zorder=5)
        ax.annotate(name, (comp, ctcvr),
                    textcoords='offset points', xytext=(10, 5),
                    fontsize=11, fontweight='bold')

ax.set_xlabel('Architecture Complexity', fontsize=12)
ax.set_ylabel('CTCVR AUC', fontsize=12)
ax.set_title('Complexity vs Performance', fontsize=13)
ax.set_xticks([1, 2, 3, 4])
ax.set_xticklabels(['Simple\nMLP', 'Shared\nTowers', 'Expert\nGating', 'Progressive\nExtraction'])
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Deployment Recommendations

> **Concept:** The choice between ESMM, MMoE, and PLE depends on your specific constraints:
> available compute, serving latency budget, and the relationship between your tasks.

In [ ]:
# Deployment recommendation summary
print("=" * 80)
print("DEPLOYMENT RECOMMENDATIONS")
print("=" * 80)

recommendations = [
    ('ESMM',
     'Best overall CTCVR AUC with simplest multi-task architecture',
     'Latency-sensitive systems, quick deployment, strong baseline'),
    ('MMoE',
     'Flexible expert routing, good when tasks have different optimal features',
     'Systems with multiple related tasks, where task relationships are complex'),
    ('PLE',
     'Reduces negative transfer with task-specific experts',
     'When tasks conflict, or when you need maximum task isolation'),
]

for model, strength, use_case in recommendations:
    ctcvr = all_results.get(model, {}).get('ctcvr_auc', 0)
    params = all_results.get(model, {}).get('params', 0)
    print(f'\n{model}:')
    print(f'  CTCVR AUC: {ctcvr:.4f}')
    print(f'  Parameters: {params:,}' if params else '  Parameters: N/A')
    print(f'  Strength: {strength}')
    print(f'  Best for: {use_case}')

# Winner determination
best_model = max(
    [(n, all_results[n].get('ctcvr_auc', 0)) for n in model_names],
    key=lambda x: x[1]
)
print(f'\n{"=" * 80}')
print(f'WINNER on Ali-CCP: {best_model[0]} (CTCVR AUC = {best_model[1]:.4f})')
print(f'{"=" * 80}')

In [ ]:
# Fig 9: Final summary visualization
fig, ax = plt.subplots(figsize=(12, 7))

# Grouped bar chart with all metrics
metrics_to_plot = ['cvr_auc', 'ctcvr_auc']
metric_labels = ['CVR AUC\n(Clicked)', 'CTCVR AUC\n(All Impressions)']

x = np.arange(len(metrics_to_plot))
width = 0.18

for i, name in enumerate(model_names):
    vals = [all_results[name].get(m, 0) for m in metrics_to_plot]
    bars = ax.bar(x + i * width, vals, width, label=name,
                  color=MODEL_COLORS[name], edgecolor='white', linewidth=1.5)
    for bar, val in zip(bars, vals):
        if val > 0:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                    f'{val:.4f}', ha='center', va='bottom', fontsize=9,
                    fontweight='bold', rotation=45)

ax.set_xticks(x + width * (len(model_names) - 1) / 2)
ax.set_xticklabels(metric_labels, fontsize=12)
ax.set_ylabel('AUC', fontsize=12)
ax.set_title('Ali-CCP Multi-Task CVR: Final Model Comparison', fontsize=15)
ax.legend(fontsize=11, loc='upper left')
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim(0.5, max(all_results[n].get('cvr_auc', 0) for n in model_names) + 0.03)

plt.tight_layout()
plt.show()

In [ ]:
# Final summary table
print("\n" + "=" * 90)
print("FINAL SUMMARY TABLE")
print("=" * 90)

summary_data = []
for name in model_names:
    r = all_results[name]
    summary_data.append({
        'Model': name,
        'CTR AUC': f'{r.get("ctr_auc", 0):.4f}' if r.get('ctr_auc') else '-',
        'CVR AUC': f'{r.get("cvr_auc", 0):.4f}',
        'CTCVR AUC': f'{r.get("ctcvr_auc", 0):.4f}',
        'Params': f'{r.get("params", 0):,}' if r.get('params') else '-',
        'SSB': 'Yes' if name != 'Naive CVR' else 'No',
    })

summary_df = pd.DataFrame(summary_data)
print(summary_df.to_string(index=False))

---

## Exercises

### Exercise 1: Ensemble Models
Create an ensemble of ESMM, MMoE, and PLE by averaging their CTCVR predictions.
Does the ensemble outperform individual models?

In [ ]:
# TODO: Exercise 1
# Hint: ensemble_pred = (esmm_ctcvr + mmoe_ctcvr + ple_ctcvr) / 3
# Compute AUC for the ensemble and compare with individual models
# Also try weighted averaging with weights tuned on a validation set

pass

### Exercise 2: Per-Field Performance Analysis
Analyze which feature fields contribute most to each model's predictions.
Compute feature importance by measuring AUC drop when each field is zeroed out.

In [ ]:
# TODO: Exercise 2
# For each field:
#   1. Zero out that field's embedding input
#   2. Re-evaluate the model
#   3. Compute AUC drop = original_AUC - zeroed_AUC
#   4. Higher drop = more important field

pass

### Exercise 3: Statistical Significance with Real Predictions
Use bootstrap resampling on actual model predictions (not simulated) to compute
95% confidence intervals. Run 1000 bootstrap samples.

In [ ]:
# TODO: Exercise 3
# Load actual predictions from saved models
# For each bootstrap sample:
#   resample test set with replacement
#   compute AUC for each model
#   collect distribution of AUC differences

pass

### Exercise 4: NDCG Comparison
Compute NDCG@K (K=10, 50, 100) for each model. Does the ranking quality differ
from AUC-based evaluation?

In [ ]:
# TODO: Exercise 4
# Implement NDCG@K using conversion labels as relevance
# Compare models at different K values

pass

---

## Summary & Key Takeaways

### What We Learned

1. **Multi-task learning is essential for CVR prediction.** The naive single-task approach trained on clicked samples suffers from Sample Selection Bias, producing systematically biased predictions when applied to all impressions. All three multi-task models (ESMM, MMoE, PLE) outperform the naive baseline on the full impression space.

2. **ESMM is the strongest performer on Ali-CCP.** With the best CTCVR AUC (~0.618), ESMM demonstrates that the simple but elegant CTR x CVR decomposition with shared embeddings is highly effective for this dataset. Reference results: Naive CVR AUC=0.5868/CTCVR=0.5385, ESMM CVR=0.6518/CTCVR=0.6177, MMoE CVR=0.6184/CTCVR=0.6066, PLE CVR=0.6284/CTCVR=0.6058.

3. **Architecture complexity does not guarantee better performance.** MMoE and PLE, despite having expert networks and gating mechanisms, did not surpass ESMM on this dataset. This suggests that when tasks (CTR and CVR) are closely related and the primary challenge is SSB rather than negative transfer, simpler architectures can win.

4. **Expert gating analysis reveals task relationships.** The MMoE gating patterns show how CTR and CVR tasks utilize different expert combinations, providing interpretability into model behavior.

5. **Deployment choice depends on constraints.** For latency-sensitive production systems, ESMM offers the best accuracy-simplicity trade-off. MMoE and PLE are better when you have multiple diverse tasks or evidence of negative transfer.

### Comparison with Tenrec Experiments

The Ali-CCP experiments validate the same multi-task learning principles observed on Tenrec, but on a more challenging industry-standard dataset with:
- Much lower positive rates (0.03% CTCVR vs ~0.6% like rate)
- Higher-dimensional sparse features (20+ fields vs 5 fields)
- Real commercial ad data vs synthetic video platform data